# Kaggle Free Tier (`GPU T4 x2` — `0 CU`) · 4-Dataset Builder & PatchCore++ Multi-Product Ablation

Runs **100% free on Kaggle (`GPU T4 x2`)** so **zero Colab Compute Units** are consumed:
1. Harmonizes **PKU-Market-PCB (`VOC_PCB` + dense `good/` normal split)**, **DeepPCB** (`1,500` template-test pairs), **VisA (`pcb1–pcb4`)**, and **Assembled PCBA Component/Solder Defects** into:
   - `unified_pcb_10cls/` (10 unified PCB/PCBA defect classes)
   - `anomaly_multiproduct/` (leak-free multi-product normal train/val/test + pixel GT masks)
2. Executes the **4-Row PatchCore++ Multi-Product Ablation**:
   - **Row A**: Baseline `WideResNet50` (`L2+L3`, `256px`, L2-norm)
   - **Row B**: Upgraded `WideResNet50` (`L1+L2`, `512px`, unnormalized contrast + **k-Center Greedy Coreset + FAISS** + `k=9` density re-weighting + border mask)
   - **Row C**: Pure Foundation ViT `DINOv2-ViT-B/14-reg` (`518px`, Product-SKU Router)
   - **Row D**: **Proposed `PatchCore++` (`dinov2_hybrid`: `DINOv2-reg + WRN50-Stem` @ `518px` + Product-SKU Router + 10-Class ROI Prototype Classifier)**
3. Saves `patchcore_plus_multiproduct.pt` and `patchcore_ablation_metrics.json` to `/kaggle/working/`.

In [ ]:
!pip install -q faiss-cpu scikit-learn scipy timm

import json, os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/Darren7218/PCB-Defect-Detection-FYP.git'
BRANCH   = 'feat/multiproduct-patchcore-v2'
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./work')
REPO_DIR = WORK_DIR / 'PCB-Defect-Detection-FYP'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import torch
print('PyTorch:', torch.__version__, '| GPUs:', torch.cuda.device_count())

In [ ]:
# Auto-discover attached Kaggle input datasets (`VOC_PCB`, `good`, `VisA`, `PCBA`) + auto-clone `DeepPCB`
KAGGLE_IN = Path('/kaggle/input')

def find_input_dir(marker_subpath: str):
    if not KAGGLE_IN.exists():
        return None
    hits = list(KAGGLE_IN.rglob(marker_subpath))
    return hits[0] if hits else None

voc_pcb_dir = find_input_dir('VOC_PCB')
good_dir    = find_input_dir('good')
visa_dir    = find_input_dir('split_csv')
if visa_dir is not None:
    visa_dir = visa_dir.parent

# Optional Assembled PCBA YOLO dataset attached in /kaggle/input (e.g. pcba-defect-detection)
pcba_dirs = [p.parent for p in KAGGLE_IN.rglob('data.yaml') if 'VOC_PCB' not in str(p)] if KAGGLE_IN.exists() else []

print('Discovered VOC_PCB :', voc_pcb_dir)
print('Discovered good/   :', good_dir)
print('Discovered VisA    :', visa_dir)
print('Discovered PCBA    :', pcba_dirs)

OUT_DATA = WORK_DIR / 'datasets'
cmd = [sys.executable, '-m', 'src.datasets.prepare_multiproduct_pcb', '--out-dir', str(OUT_DATA), '--download-deeppcb']
if voc_pcb_dir: cmd += ['--voc-pcb', str(voc_pcb_dir)]
if good_dir:    cmd += ['--good-boards', str(good_dir)]
if visa_dir:    cmd += ['--visa', str(visa_dir)]
if pcba_dirs:   cmd += ['--pcba', *[str(x) for x in pcba_dirs]]

subprocess.run(cmd, check=True)
print(json.loads((OUT_DATA / 'dataset_manifest.json').read_text()))

In [ ]:
# Collect Multi-Product Anomaly Benchmark SKUs (`train/good`, `val/good`, `test/...`)
from src.anomaly.patchcore_plus import PatchCorePlus, find_images

ANOM_ROOT = OUT_DATA / 'anomaly_multiproduct'
sku_train_normals = {}
sku_dirs = []
for sku_dir in sorted(ANOM_ROOT.iterdir()):
    t_good = sku_dir / 'train' / 'good'
    if sku_dir.is_dir() and t_good.exists():
        imgs = find_images([t_good])
        if imgs:
            # Cap per-SKU normal tiles to 250 for fast, balanced multi-product memory bank construction
            sku_train_normals[sku_dir.name] = imgs[:250]
            sku_dirs.append(sku_dir)

print(f'Active PCB Product SKUs ({len(sku_train_normals)}):', {k: len(v) for k, v in sku_train_normals.items()})

In [ ]:
# Run 4-Row PatchCore++ Multi-Product Ablation Study
ABLATION_CONFIGS = [
    ('RowA_WRN50_256_L2Norm_Baseline', dict(image_size=256, backbone_name='wide_resnet50_2', cnn_layers=('layer2', 'layer3'), l2_normalize_cnn=True,  knn_k=1, coreset_ratio=0.02)),
    ('RowB_WRN50_512_GreedyCoreset',   dict(image_size=512, backbone_name='wide_resnet50_2', cnn_layers=('layer1', 'layer2'), l2_normalize_cnn=False, knn_k=9, coreset_ratio=0.04)),
    ('RowC_DINOv2_Reg_518_PureViT',    dict(image_size=518, backbone_name='dinov2_vitb14_reg', cnn_layers=('layer1', 'layer2'), l2_normalize_cnn=False, knn_k=9, coreset_ratio=0.05)),
    ('RowD_PatchCorePlus_DINOv2_Hybrid', dict(image_size=518, backbone_name='dinov2_hybrid',   cnn_layers=('layer1', 'layer2'), l2_normalize_cnn=False, knn_k=9, coreset_ratio=0.05)),
]

ablation_report = {}
best_model = None

for row_name, cfg in ABLATION_CONFIGS:
    print('\n' + '=' * 68)
    print('Evaluating:', row_name)
    c_ratio = cfg.pop('coreset_ratio')
    model = PatchCorePlus(**cfg)
    model.fit(sku_train_normals, batch_size=8, coreset_ratio=c_ratio)

    per_sku_metrics = {}
    img_aurocs, pix_aurocs = [], []
    for s_dir in sku_dirs:
        t_dir = s_dir / 'test'
        m_dir = s_dir / 'ground_truth' / 'test'
        if not t_dir.exists():
            continue
        res = model.evaluate_anomaly_dir(t_dir, m_dir if m_dir.exists() else None, product_sku='auto')
        per_sku_metrics[s_dir.name] = res
        if res['image_auroc'] is not None: img_aurocs.append(res['image_auroc'])
        if res['pixel_auroc_mean'] is not None: pix_aurocs.append(res['pixel_auroc_mean'])

    summary = {
        'mean_image_auroc': sum(img_aurocs) / len(img_aurocs) if img_aurocs else None,
        'mean_pixel_auroc': sum(pix_aurocs) / len(pix_aurocs) if pix_aurocs else None,
        'per_sku': per_sku_metrics,
    }
    ablation_report[row_name] = summary
    print(f"{row_name} -> Mean Image-AUROC: {summary['mean_image_auroc']} | Mean Pixel-AUROC: {summary['mean_pixel_auroc']}")
    if row_name == 'RowD_PatchCorePlus_DINOv2_Hybrid':
        best_model = model

(WORK_DIR / 'patchcore_ablation_metrics.json').write_text(json.dumps(ablation_report, indent=2))
print('Saved ablation metrics ->', WORK_DIR / 'patchcore_ablation_metrics.json')

In [ ]:
# Fit 10-Class Few-Shot Defect Prototype Bank onto Best PatchCore++ Model & Export Checkpoint
from PIL import Image
from src.datasets.taxonomy import UNIFIED_CLASS_NAMES

DET_ROOT = OUT_DATA / 'unified_pcb_10cls'
labeled_samples = []
train_img_dir = DET_ROOT / 'images' / 'train'
train_lbl_dir = DET_ROOT / 'labels' / 'train'

if best_model is not None and train_img_dir.exists():
    for img_p in sorted(train_img_dir.iterdir())[:1200]:
        lbl_p = train_lbl_dir / f'{img_p.stem}.txt'
        if not lbl_p.exists():
            continue
        w, h = Image.open(img_p).size
        boxes = []
        for line in lbl_p.read_text().splitlines():
            parts = line.split()
            if len(parts) != 5: continue
            cid = int(float(parts[0]))
            cx, cy, bw, bh = map(float, parts[1:])
            boxes.append((UNIFIED_CLASS_NAMES[cid], (cx - bw/2)*w, (cy - bh/2)*h, (cx + bw/2)*w, (cy + bh/2)*h))
        if boxes:
            labeled_samples.append((img_p, boxes))

    proto_counts = best_model.fit_defect_prototypes(labeled_samples, max_per_class=200)
    print('Fitted Defect Prototype Counts:', proto_counts)
    ckpt_out = WORK_DIR / 'patchcore_plus_multiproduct.pt'
    best_model.save(ckpt_out)
    print('Saved production PatchCore++ checkpoint ->', ckpt_out)